In [ ]:
import pandas as pd

In [ ]:
pd.set_option('display.max_colwidth', None) # to increase the width of the columns

In [ ]:
# orders.csv
url = "https://drive.google.com/file/d/1Vu0q91qZw6lqhIqbjoXYvYAQTmVHh6uZ/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orders = pd.read_csv(path)

# orderlines.csv
url = "https://drive.google.com/file/d/1FYhN_2AzTBFuWcfHaRuKcuCE6CWXsWtG/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orderlines = pd.read_csv(path)

Before we begin, let's create a copy of the `orders` and `orderlines` DataFrames. This way we are sure any of our changes won't affect the original DataFrames.

In [ ]:
orders_df = orders.copy()

In [ ]:
orderlines_df = orderlines.copy()

## 1.&nbsp; Duplicates


In [ ]:
# orders
orders_df.duplicated().sum()

np.int64(0)

In [ ]:
# orderlines
orderlines_df.duplicated().sum()

np.int64(0)

# 2.&nbsp; `.info()`

In [ ]:
orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      226909 non-null  int64  
 1   created_date  226909 non-null  object 
 2   total_paid    226904 non-null  float64
 3   state         226909 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 6.9+ MB


In [ ]:
orderlines_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   id                293983 non-null  int64 
 1   id_order          293983 non-null  int64 
 2   product_id        293983 non-null  int64 
 3   product_quantity  293983 non-null  int64 
 4   sku               293983 non-null  object
 5   unit_price        293983 non-null  object
 6   date              293983 non-null  object
dtypes: int64(4), object(3)
memory usage: 15.7+ MB


* `date` should be a datetime datatype
* `unit_price` should be a float datatype

## 3.&nbsp; Missing values

### 3.1.&nbsp; Orders
* `total_paid` has 5 missing values

In [ ]:
print(f"5 missing values represents {((orders_df.total_paid.isna().sum() / orders_df.shape[0])*100).round(5)}% of the rows in our DataFrame")

5 missing values represents 0.0022% of the rows in our DataFrame


In [ ]:
orders_df.total_paid.isna().value_counts(normalize=True)

,proportion
total_paid,
False,0.999978
True,0.000022


As there is such a tiny amount of missing values, we will simply delete these rows, as we have enough data without them.

In [ ]:
orders_df = orders_df.loc[~orders.total_paid.isna(), :]

### 3.2.&nbsp; Orderlines
There are no missing values in `orderlines`

## 4.&nbsp; Datatypes

### 4.1.&nbsp; Orders
* `created_date` should become datetime datatype

In [ ]:
orders_df["created_date"] = pd.to_datetime(orders_df["created_date"])

### 4.1.&nbsp; Orderlines


In [ ]:
orderlines_df["date"] = pd.to_datetime(orderlines_df["date"])

#### 4.1.2.&nbsp;`unit_price`

In [ ]:
orderlines_df.unit_price.str.contains(r"\d+\.\d+\.\d+").value_counts()

,count
unit_price,
False,257814
True,36169


Looks like over 36000 rows in `orderlines` are affected by this problem. Let's work out how much that is as a percentage of our total data.

In [ ]:
two_dot_percentage = ((orderlines_df.unit_price.str.contains(r"\d+\.\d+\.\d+").value_counts().iloc[1] / orderlines_df.shape[0])*100).round(2)
print(f"The 2 dot problem represents {two_dot_percentage}% of the rows in our DataFrame")

The 2 dot problem represents 12.3% of the rows in our DataFrame


In [ ]:
two_dot_order_ids_list = orderlines_df.loc[orderlines_df.unit_price.str.contains(r"\d+\.\d+\.\d+"), "id_order"]

orderlines_df = orderlines_df.loc[~orderlines_df.id_order.isin(two_dot_order_ids_list)]

In [ ]:
orderlines_df.shape[0]

216250

We still have 216250 rows in orderlines to work with. This should be more than enough for our evaluation.

Now that all of the 2 decimal point prices have been removed, let's try again to convert the column `unit_price` to the correct datatype.

In [ ]:
orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"])

It worked perfectly

In [ ]:
# products.csv
url = "https://drive.google.com/file/d/1afxwDXfl-7cQ_qLwyDitfcCx3u7WMvkU/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
products = pd.read_csv(path)

In [ ]:
products_df = products.copy()

In [ ]:
products_df.head()

,sku,name,desc,price,promo_price,in_stock,type
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,499.899,1,8696
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59,589.996,0,13855401
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59,569.898,0,1387
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25,229.997,0,1230
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pro (2006/07/08).,34.99,31.99,1,1364


In [ ]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19326 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sku          19326 non-null  object
 1   name         19326 non-null  object
 2   desc         19319 non-null  object
 3   price        19280 non-null  object
 4   promo_price  19326 non-null  object
 5   in_stock     19326 non-null  int64 
 6   type         19276 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.0+ MB


We'll go through the steps above in order
* Duplicates
* Missing values
* Datatypes

But I think we can all see straight away from `products.head()` above that some of the prices in `promo_price` look wrong. Let's make sure we deal with this later.

## Duplicates

In [ ]:
products_df.duplicated().sum()

np.int64(8746)

In [ ]:
products_df = products_df.drop_duplicates()

## `.info()`

In [ ]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10580 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sku          10580 non-null  object
 1   name         10580 non-null  object
 2   desc         10573 non-null  object
 3   price        10534 non-null  object
 4   promo_price  10580 non-null  object
 5   in_stock     10580 non-null  int64 
 6   type         10530 non-null  object
dtypes: int64(1), object(6)
memory usage: 661.2+ KB


### Missing values
We can see from `.info()` above that we have missing values in `desc` and `price`

#### `desc`

In [ ]:
products_df["desc"].isna().sum()

np.int64(7)

In [ ]:
products_df.loc[products_df['desc'].isna(), :]

,sku,name,desc,price,promo_price,in_stock,type
16126,WDT0211-A,"Open - Purple 2TB WD 35 ""PC Security Mac hard drive and NAS",NaN,107,814.659,0,1298
16128,APP1622-A,"Open - Apple Smart Keyboard Pro Keyboard Folio iPad 9.7 """,NaN,1.568.206,1.568.206,0,1298
17843,PAC2334,Synology DS718 + NAS Server | 10GB RAM,NaN,566.35,5.659.896,0,12175397
18152,KAN0034-A,"Open - Kanex USB-C Gigabit Ethernet Adapter MacBook 12 """,NaN,29.99,237.925,0,1298
18490,HTE0025,Hyper Pearl 1600mAh battery Mini USB Mirror and Comic Blond,NaN,24.99,22.99,1,1515
18612,OTT0200,OtterBox External Battery Power Pack 20000 mAHr,NaN,79.99,56.99,1,1515
18690,HOW0001-A,Open - Honeywell thermostat Lyric zonificador T6 Intelligent Wireless (cable),NaN,199.99,1.441.174,0,11905404


In [ ]:
products_df.loc[products_df['desc'].isna(), 'desc'] = products_df.loc[products_df['desc'].isna(), 'name']

In [ ]:
products_df.loc[products_df['desc'].isna(), :]

,sku,name,desc,price,promo_price,in_stock,type


#### `price`

In [ ]:
products_df.price.isna().sum()

np.int64(46)

In [ ]:
print(f"The missing values in price are {(products_df.price.isna().value_counts(normalize=True).iloc[1] * 100).round(2)}% of all rows in the DataFrame")

The missing values in price are 0.43% of all rows in the DataFrame


Option 1: `.loc`

In [ ]:
products_df = products_df.loc[~products['price'].isna()]

Option 2: `.dropna()`

#### `type`

In [ ]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10534 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sku          10534 non-null  object
 1   name         10534 non-null  object
 2   desc         10534 non-null  object
 3   price        10534 non-null  object
 4   promo_price  10534 non-null  object
 5   in_stock     10534 non-null  int64 
 6   type         10484 non-null  object
dtypes: int64(1), object(6)
memory usage: 658.4+ KB


### Data types

We saw from looking at the output of `.info()` that both `price` and `promo_price` have been stored as objects and not as a numerical datatypes. We also saw while solving other problems that both columns have some prices with 3 decimal places and others with 2 decimal points - the latter will prevent us from converting the datatype to numerical, so first we must investigate and solve these problems.

#### `price`

First, let's see how many values are affected by the 2-decimal-dot problems or 3 decimal places.

In [ ]:
price_problems_number = products_df.loc[(products_df.price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.price.str.contains(r"\d+\.\d{3,}")), :].shape[0]
price_problems_number

542

In [ ]:
print(f"The column price has in total {price_problems_number} wrong values. This is {round(((price_problems_number / products_df.shape[0]) * 100), 2)}% of the rows of the DataFrame")

The column price has in total 542 wrong values. This is 5.15% of the rows of the DataFrame


5.15% is a reasonable amount of our data. However, the price column will be important to understanding discounts, so I'd like it to be very trustworthy as we are basing business decisions on it. Therefore, we'll delete these rows

In [ ]:
products_df = products_df.loc[~((products_df.price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.price.str.contains(r"\d+\.\d{3,}"))), :]

In [ ]:
products_df.sample(50)

,sku,name,desc,price,promo_price,in_stock,type
17905,PAC2280,Synology DS918 + NAS Server | 16GB | 32TB (4x8TB) WD Red,NAS server of the Plus Series for companies seeking high performance and the ability to scale memory and,2208.71,17.213.678,0,12175397
1687,PAC1800,Synology DS1815 Pack + | 16GB RAM | Seagate 16TB IronWolf,Nas + DS1815 RAM with capacity 16GB and 16TB (8x2TB) IronWolf Seagate Drives for Mac and PC,1974.49,17.927.457,0,12175397
14553,PAC1547,Enlargement Kit 2TB SSD Crucial MX300 + RAM + 8GB 1333MHz Datadoubler MacBook Pro 2011,SSD upgrade kit 2TB + 8GB 1333MHz RAM for MacBook Pro Early / Late2011 tools,796.52,6.405.849,0,1433
18523,TIL0010,Tile Combo Bluetooth Locators (Pack 4 units),Tile Combo pack 4 Bluetooth locators for your iPhone iPad and Apple Watch,74.99,579.905,1,11905404
1589,SPE0133,"Speck SeeThru MacBook Pro Retina Clear housing 15 ""transparent",Protective housing polycarbonate Macbooks Retina 15 inches.,49.9,399.905,0,13835403
12603,LUN0016,Flak lunatik Case iPhone 6 / 6S Silver,Multilayer Polycarbonate and silicone iPhone 6 / 6s,34.95,22.99,0,11865403
18823,KOO0007,Koogeek Homekit Smart Plug socket with Siri voice control,Koogeek plug with Apple HomeKit technology and Siri voice control for your home,34.99,289.899,0,11905404
10647,OTT0125,Otterbox iPad Mini Folio Symmetry 1/2/3 Black,Protective Case for iPad Mini Folio format 1/2/3 Black.,79.99,469.904,0,12635403
17592,SAN0188,SanDisk Ultra 64GB microSDXC card A1,Micro Memory Card with adapter read speed 100MB / s and video speed UHS U1,32.99,290.001,1,57445397
13001,IOT0020,iOttie Active Edge Support iPhone Indigo Blue Bicycle,Bike mount and bars up to 3 cm for iPhone,39.99,279.897,0,5720


In [ ]:
products_df["price"] = pd.to_numeric(products_df["price"])

In [ ]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9992 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sku          9992 non-null   object 
 1   name         9992 non-null   object 
 2   desc         9992 non-null   object 
 3   price        9992 non-null   float64
 4   promo_price  9992 non-null   object 
 5   in_stock     9992 non-null   int64  
 6   type         9946 non-null   object 
dtypes: float64(1), int64(1), object(5)
memory usage: 624.5+ KB


#### `promo_price`

Again, let's begin by seeing how many values are affected by the 2-decimal-dots problem, or the 3 decimal-places problem

In [ ]:
promo_problems_number = products_df.loc[(products_df.promo_price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.promo_price.str.contains(r"\d+\.\d{3,}")), :].shape[0]
promo_problems_number

9232

In [ ]:
print(f"The column promo_price has in total {promo_problems_number} wrong values. This is {round(((promo_problems_number / products_df.shape[0]) * 100), 2)}% of the rows of the DataFrame")

The column promo_price has in total 9232 wrong values. This is 92.39% of the rows of the DataFrame


In [ ]:
promo_price_df = products_df.loc[(products_df.promo_price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.promo_price.str.contains(r"\d+\.\d{3,}")), :]
promo_price_df.sample(50)

,sku,name,desc,price,promo_price,in_stock,type
18372,BEL0340,Belkin Cable HDMI High Speed ​​with Ethernet 5m,Belkin Cable High Speed ​​HDMI - Ethernet 5m for Mac,29.99,269.903,1,1325
2621,OWC0127,OWC Mercury Extreme Pro SSD 240GB 6GB,SSD 240GB SATA hard drive for Mac and PC III.,217.99,1.705.834,0,12215397
2972,LIF0083,Lifeproof nüüd Case Waterproof iPhone 6 Plus White,waterproof case for extreme conditions and iPhone 6 Plus.,99.99,249.865,0,11865403
17272,TUC0327,"Tucano tugo Medium Backpack 20L MacBook Pro 13 ""and 15"" / Retina / Air 13 ""Blue",Backpack 20 liter single compartment handles and pockets for MacBook Pro 13-inch and 15-inch / Retina / Air 13-inch,59.90,299.899,0,1392
11071,PAC1631,Pack QNAP TS-251 + | 2GB RAM | Seagate NAS 16TB,RAM nas TS-251 + 2 GB memory + 12TB (2x6TB) Hard Seagate IronWolf for Mac and PC,908.98,7.111.787,1,12175397
1373,APP0925,Apple Mac mini Core i7 3GHz | 16GB RAM | 256GB Flash,PC Mac mini Core i7 3GHz 16GB 1TB 256GB Flash (MGEQ2YP / A).,1579.00,15.000.043,0,1282
422,GRT0264,Griffin CinemaSeat Case for iPad mini / black Retina,IPad mini car cover neoprene support.,39.99,189.897,0,12635403
11871,MAC0125,Macally USB 3.0 to Gigabit Ethernet Adapter,Cable USB 3.0 Gigabit Ethernet adapter to connect 10/100 / 1000Mbps.,34.95,249.865,1,12585395
1475,SPH0012,Sphero Ollie Robot Blue,Smartoy of Sphero Ollie Robot remote control via Bluetooth App.,109.95,1.049.905,1,11905404
2462,FIF0009,FiftyThree Walnut Wood Pencil Pointer iPad,digital pointer with Bluetooth and multiplicity of stroke for iPad.,59.99,569.898,0,1229


So we were correct, over 90% of the data in this column is corrupt. There's no point deleting all of these rows, then we would barely have a products table. Instead, as it's only this column that appears to be very untrustworthy, we will delete the column.

In [ ]:
products_cl = products_df.drop(columns=["promo_price"])

In [ ]:
products_cl.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9992 entries, 0 to 19325
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   sku       9992 non-null   object 
 1   name      9992 non-null   object 
 2   desc      9992 non-null   object 
 3   price     9992 non-null   float64
 4   in_stock  9992 non-null   int64  
 5   type      9946 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 546.4+ KB


In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define the project folder
output_folder = "/content/drive/MyDrive/Project 2 (Discounts) Datasets"

# Save cleaned DataFrames to Google Drive
orders_df.to_csv(f"{output_folder}/orders_cl.csv", index=False)
orderlines_df.to_csv(f"{output_folder}/orderlines_cl.csv", index=False)
products_cl.to_csv(f"{output_folder}/products_cl.csv", index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
